# 5 · Building a Custom Agent
Build → Observe → Evaluate → Diagnose → Improve.
Configura nombre, objetivo, capacidades y límites del perfil; justifica qué decisión delegas y qué comprueba Python.
El perfil se utiliza en el mismo runtime de S3/S4. La mejora verificable afecta al contrato de aplicación; no se atribuye al modelo una mejora sintética.

Predice el resultado antes de ejecutar y anota tus observaciones.

In [ ]:
import json
import os
import sys
from pathlib import Path

# Cada notebook empieza desde datos preparados, sin archivos de sesiones anteriores.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
from dotenv import load_dotenv

if not os.getenv("CARRITO_NOTEBOOK_CHECK"):
    load_dotenv(ROOT / ".env")
from carrito.store import create_store
from carrito.tools import StoreTools

db = create_store()
tools = StoreTools(db, user_id="user1")  # Identidad fijada por el host.
RUN_LIVE = False  # Cambiar explícitamente a True permite llamadas de pago.
COMPLETION = {}

def check(name, condition):
    COMPLETION[name] = bool(condition)
    print(("OK" if condition else "PENDIENTE") + ": " + name)

print("MODO OFFLINE: fixtures deterministas. No miden calidad del LLM.")

In [ ]:
from carrito.context import validate_recommendation
from carrito.lab import (
    AgentProfile,
    catalog_response,
    compare_contracts,
    run_profile,
    trace_metrics,
)
from carrito.model import FixtureModel
from carrito.skills import activate_skill

profile = AgentProfile(name="budget-shopping", instructions="Compara opciones dentro del presupuesto. Si no hay productos, no inventes alternativas.", allowed_tools=("search_products", "search_policies"), max_steps=4, enable_skills=False)
print(profile)
print("Contrato baseline sin productos:", catalog_response([], improved=False))

## Skills: procedimiento y contexto
Inspecciona metadata y procedimiento de `examples/skills/return-help/SKILL.md`.
Compara prompt, Skill, tool, MCP server y host. Aquí la activación por palabras es deliberadamente simple. Activar la Skill no habilita tools ni confirma escrituras.

In [ ]:
for prompt in ["Busco auriculares", "Quiero devolver el pedido 104"]:
    activation = activate_skill(prompt)
    print(prompt, activation)
from dataclasses import replace

return_profile = replace(profile, name="returns-guide", enable_skills=True, allowed_tools=("get_order", "search_policies"))
return_run = run_profile("Quiero devolver el pedido 104", tools, FixtureModel.for_scenario("return"), return_profile)
print([(e["type"], e.get("name"), e.get("result")) for e in return_run["events"] if e["type"] in {"skill_activated", "tool_result"}])
assert tools.pending is None

## Dataset y criterio de éxito
Antes de ejecutar: define expected behavior para resultado vacío, producto fuera de presupuesto, ID desconocido y tool denegada. Separa calidad de respuesta y invariantes del host. Escribe dos casos propios antes de modificar el sistema.

In [ ]:
own_cases = [
    {"question": "", "expected_behavior": "", "automatic_check": "", "human_review": ""},
    {"question": "", "expected_behavior": "", "automatic_check": "", "human_review": ""},
]
from carrito.evaluation import load_cases

print([{k: v for k, v in case.items() if k != "fixture_calls"} for case in load_cases("dev")])
# No abrir final hasta congelar la mejora.

## Leer una traza
Localiza request, model response, tool result y response_contract. Distingue respuesta original del modelo de respuesta final de aplicación. Cuenta llamadas y fallos antes de proponer un cambio.

In [ ]:
baseline_report = compare_contracts(profile=profile)
empty_trace = baseline_report["before"][1]["trace"]
print(json.dumps(empty_trace, ensure_ascii=False, indent=2))
print("Métricas:", trace_metrics(empty_trace))
# Tokens fixture=0; coste desconocido. No equivalen a un modelo gratis o instantáneo.

## Baseline y diagnóstico
Ejecuta los cuatro casos, clasifica primera causa y contrasta con la tool. El catálogo vacío es un resultado válido; el contrato `recommend` sin IDs es el fallo. Añade un caso donde hay datos pero una explicación textual miente: el checker básico no lo detecta.

In [ ]:
for case in baseline_report["before"]:
    print(case["query"], case["passed"], case["checks"])
diagnosis = {"symptom": "", "first_bad_event": "", "owner": "", "hypothesis": "", "test": ""}
print("Diagnóstico a completar:", diagnosis)
print("Baseline:", baseline_report["before_passed"], "/", baseline_report["total"])

## Cambiar una pieza del sistema
Completa un constructor de respuesta sobre productos observados. Preserva IDs, distingue `recommend`/`no_match` y solicita una preferencia alternativa cuando no hay resultados. Pásalo a `compare_contracts`: el mismo runtime ejecuta ambas versiones. Tu mejora no puede copiar outputs esperados ni tocar fixtures.

In [ ]:
def improved_response(products):
    # TODO: contrato coherente tanto para catálogo vacío como para productos reales.
    return catalog_response(products, improved=False)

In [ ]:
report = compare_contracts(builder=improved_response, profile=profile, mode="live" if RUN_LIVE else "fixture")
check("resuelve el fallo observado", report["after_passed"] > report["before_passed"])
check("conserva casos válidos", all(row["passed"] for row in report["after"]))
print("ANTES / DESPUÉS:", report["before_passed"], report["after_passed"], "/", report["total"])
print("¿Calidad del modelo medida?", report["model_quality_measured"])

## Human review y judge
Etiqueta los ejemplos sin mirar la etiqueta del juez. Después inspecciona desacuerdo y evidencia. Tres ejemplos no calibran un judge. Los jueces también fallan; una rúbrica acotada y un humano permiten diagnosticar por qué.

In [ ]:
from carrito.judge import judge_demo

judgments = judge_demo(mode="live" if RUN_LIVE else "fixture")
print(json.dumps(judgments, ensure_ascii=False, indent=2))
print("Revisión humana: ¿la explicación final está sustentada por el resultado de tool?")

## Regression run y observabilidad
Ejecuta los casos dev del backend, compara tus cuatro recorridos y revisa dos respuestas finales. Define dos casos propios ejecutables. Registra perfil, cambio, casos, límites y alcance de la evidencia. Un pase de tools/estado no garantiza que la respuesta sea correcta.

In [ ]:
from carrito.evaluation import run_evals

backend = run_evals("dev", mode="fixture")
print("Invariantes backend:", backend["passed"], "/", backend["total"])
for old, new in zip(report["before"], report["after"], strict=True):
    print({"query": new["query"], "before": old["passed"], "after": new["passed"], "model_calls": new["trace"]["metrics"]["model_calls"], "tool_errors": new["trace"]["metrics"]["tool_errors"]})
# Casos propios deben probar TU función y los resultados observados.
for products in [[], tools.search_products("teclado", 60)]:
    print(validate_recommendation(improved_response(products), products, 60))
# Define casos adicionales antes de modificar la implementación.
additional_cases = [("bicicleta", 100), ("altavoz", 60)]
additional_report = compare_contracts(
    builder=improved_response, profile=profile, cases=additional_cases
)
print("Casos adicionales:", additional_report["after_passed"], "/", additional_report["total"])
print("Backend y contrato se reportan por separado; no miden calidad lingüística.")

## Entrega
Perfil propio, criterio de éxito, dos casos propios, resultado antes/después, dos trazas comentadas y una limitación. Define qué monitorizarías online y cuándo pararías un despliegue. El trabajo termina cuando puedes explicar si funciona y mejorarlo con evidencia.

In [ ]:
print(json.dumps(COMPLETION, ensure_ascii=False, indent=2))
print("CHECKPOINT_COMPLETO" if all(COMPLETION.values()) else "Completa las celdas TODO y repite los checks.")